# 🧪 Synthetic Data Generation (Using sdg_hub)

This notebook uses **sdg_hub** to synthetically generate training data
for the τ-Knowledge banking_knowledge domain.

## Generation Types

| Type | Ratio | Description |
|------|-------|-------------|
| Policy QA | ~30% | Q&A on policy conditions, restrictions, and exceptions |
| Policy Application | ~25% | Document + customer situation → allowed/denied/needs confirmation |
| Tool Selection | ~20% | Current observation + tools → next tool call |
| Trajectory | ~15% | Conversation + tool observations → action sequence |
| Clarification | ~10% | Incomplete situation → follow-up question |

## Profiles
- **smoke**: 64 samples (for quick validation)
- **lab**: 2,000–5,000 samples (for training)

> ⚠️ **A teacher model endpoint is required** (SDG_TEACHER_ENDPOINT)  
> This notebook is part of the **authoring path**, not the learner path.  
> Learners use the prepared bundle directly without this process.

In [ ]:
"""Load SDG config and check teacher endpoint."""

import os
from pathlib import Path

from rhoai_model_training_lab.config import load_env, load_yaml_config, PROJECT_ROOT

load_env()

sdg_config = load_yaml_config("configs/sdg.yaml")

# Teacher endpoint check
teacher_endpoint = os.environ.get("SDG_TEACHER_ENDPOINT", "")
teacher_model = os.environ.get("SDG_TEACHER_MODEL", "")
teacher_api_key = os.environ.get("SDG_TEACHER_API_KEY", "")

print("=" * 70)
print("🔧 SDG Configuration and Teacher Model Check")
print("=" * 70)
print(f"  Pipeline: {sdg_config['pipeline']['name']}")
print(f"  Version: {sdg_config['pipeline']['version']}")
print(f"  Checkpointing: {sdg_config['pipeline']['checkpointing']}")
print(f"  Seed: {sdg_config['generation']['seed']}")
print()

print("--- Teacher Model ---")
if teacher_endpoint:
    print(f"  ✅ Endpoint: {teacher_endpoint}")
    print(f"     Model: {teacher_model}")
    print(f"     API key: {'✅ set' if teacher_api_key else '❌ not set'}")
    print(f"     Max concurrency: {sdg_config['teacher']['max_concurrent']}")
    print(f"     Timeout: {sdg_config['teacher']['timeout_seconds']}s")
    print(f"     Budget limit: ${sdg_config['teacher']['budget_limit_usd']}")

    # Test connectivity
    try:
        import httpx
        resp = httpx.get(f"{teacher_endpoint}/models", timeout=10,
                        headers={"Authorization": f"Bearer {teacher_api_key}"} if teacher_api_key else {})
        if resp.status_code == 200:
            print(f"  ✅ Connection successful")
        else:
            print(f"  ⚠️  HTTP {resp.status_code}")
    except Exception as exc:
        print(f"  ⚠️  Connection test failed: {exc}")
else:
    print("  ❌ SDG_TEACHER_ENDPOINT is not set.")
    print("     Set the teacher model endpoint in the .env file.")
    print("     A teacher model is required for synthetic data generation.")

# Show target counts
print("\n--- Target Sample Counts ---")
for profile_name in ["smoke", "lab"]:
    counts = sdg_config["generation"]["target_counts"].get(profile_name, {})
    print(f"  {profile_name}: {counts.get('total', '?')} total")
    for stype, count in counts.items():
        if stype != "total":
            print(f"    {stype}: {count}")

In [ ]:
"""Generate smoke profile first (64 samples)."""

import subprocess
import sys

print("=" * 70)
print("🧪 Smoke Profile Generation (64 samples)")
print("=" * 70)

if not teacher_endpoint:
    print("❌ Teacher model endpoint not set — skipping generation.")
    print("   Set SDG_TEACHER_ENDPOINT in your .env file.")
else:
    generate_script = PROJECT_ROOT / "scripts" / "generate_synthetic.py"

    if generate_script.exists():
        cmd = [
            sys.executable, str(generate_script),
            "--config", "configs/sdg.yaml",
            "--profile", "smoke",
        ]
        print(f"  Running: {' '.join(cmd)}")
        print("  (Teacher model calls in progress...)\n")

        result = subprocess.run(
            cmd, capture_output=True, text=True, cwd=str(PROJECT_ROOT),
        )

        if result.returncode == 0:
            print("✅ Smoke profile generation completed")
            print(result.stdout[-500:] if len(result.stdout) > 500 else result.stdout)
        else:
            print(f"❌ Generation failed (exit code: {result.returncode})")
            print(result.stderr[-500:] if len(result.stderr) > 500 else result.stderr)
    else:
        print("  ⚠️  generate_synthetic.py script not found.")
        print("  Manual execution:")
        print("    python scripts/generate_synthetic.py --config configs/sdg.yaml --profile smoke")

        # Try using sdg_hub directly
        print("\n  Or use the sdg_hub API directly:")
        print("    from sdg_hub import generate")
        print("    generate(config='configs/sdg.yaml', profile='smoke')")

In [ ]:
"""Run validation on generated data."""

print("=" * 70)
print("✅ Generated Data Validation")
print("=" * 70)

canonical_path = PROJECT_ROOT / sdg_config["output"]["canonical_path"]
rejected_path = PROJECT_ROOT / sdg_config["output"]["rejected_path"]

if canonical_path.exists():
    canonical_files = list(canonical_path.glob("*.jsonl"))
    print(f"  Canonical data files: {len(canonical_files)}")

    total_accepted = 0
    for cf in canonical_files:
        with open(cf) as f:
            count = sum(1 for line in f if line.strip())
        total_accepted += count
        print(f"    {cf.name}: {count} samples")
    print(f"  Total accepted: {total_accepted}")

    # Check rejected
    if rejected_path.exists():
        rejected_files = list(rejected_path.glob("*.jsonl"))
        total_rejected = 0
        for rf in rejected_files:
            with open(rf) as f:
                count = sum(1 for line in f if line.strip())
            total_rejected += count
        print(f"  Total rejected: {total_rejected}")
        if total_accepted + total_rejected > 0:
            rate = total_accepted / (total_accepted + total_rejected) * 100
            print(f"  Acceptance rate: {rate:.1f}%")

    # Run validation script
    validate_script = PROJECT_ROOT / "scripts" / "validate_synthetic.py"
    if validate_script.exists():
        cmd = [
            sys.executable, str(validate_script),
            "--config", "configs/data-preparation.yaml",
        ]
        print(f"\n  Running validation: {' '.join(cmd)}")
        result = subprocess.run(cmd, capture_output=True, text=True, cwd=str(PROJECT_ROOT))
        if result.returncode == 0:
            print("  ✅ Validation passed")
            print(result.stdout[-300:] if len(result.stdout) > 300 else result.stdout)
        else:
            print(f"  ❌ Validation failed")
            print(result.stderr[-300:] if len(result.stderr) > 300 else result.stderr)
else:
    print(f"  ⚠️  Canonical data path not found: {canonical_path}")
    print("     Please generate the smoke profile first.")

In [ ]:
"""Review quality report."""

import json

print("=" * 70)
print("📊 Quality Report Review")
print("=" * 70)

# Check validation config
val_config = sdg_config.get("validation", {})
print("Validation settings:")
print(f"  Independent validator: {val_config.get('independent_validator', False)}")
print(f"  Schema validation: {val_config.get('schema_validation', False)}")
print(f"  Grounding check: {val_config.get('grounding_check', False)}")
print(f"  Condition check: {val_config.get('condition_check', False)}")
print(f"  Tool schema check: {val_config.get('tool_schema_check', False)}")
print(f"  Simulation replay: {val_config.get('simulation_replay', False)}")
print(f"  Deduplication: {val_config.get('deduplication', {})}")
print(f"  Contamination check: {val_config.get('contamination_check', {})}")

# Look for quality reports
usage_path = PROJECT_ROOT / sdg_config["output"]["usage_accounting_path"]
if usage_path.exists():
    with open(usage_path) as f:
        usage = json.load(f)
    print(f"\nTeacher model usage:")
    for k, v in usage.items():
        print(f"  {k}: {v}")

logs_path = PROJECT_ROOT / sdg_config["output"]["logs_path"]
if logs_path.exists():
    log_files = list(logs_path.glob("*.log")) + list(logs_path.glob("*.jsonl"))
    print(f"\nLog files: {len(log_files)}")
    for lf in log_files[:5]:
        print(f"  {lf.name}")

print("\n  ℹ️  Record whether the validator model and generator model are the same in the quality review.")
print("     LLM-only judgments do not constitute complete validation.")

In [ ]:
"""Generate lab profile (2000-5000 samples)."""

print("=" * 70)
print("🏭 Lab Profile Generation (2,000–5,000 samples)")
print("=" * 70)

lab_counts = sdg_config["generation"]["target_counts"].get("lab", {})
print(f"  Target total samples: {lab_counts.get('total', '?')}")
for stype, count in lab_counts.items():
    if stype != "total":
        pct = count / lab_counts.get('total', 1) * 100
        print(f"    {stype}: {count} (~{pct:.0f}%)")

print("\n⚠️  These targets are accepted sample counts, not guaranteed output.")
print("   Actual generation count may be higher depending on acceptance rate.")
print(f"   Budget limit: ${sdg_config['teacher']['budget_limit_usd']}")

if not teacher_endpoint:
    print("\n❌ Teacher model endpoint not set — skipping generation.")
else:
    generate_script = PROJECT_ROOT / "scripts" / "generate_synthetic.py"
    if generate_script.exists():
        cmd = [
            sys.executable, str(generate_script),
            "--config", "configs/sdg.yaml",
            "--profile", "lab",
        ]
        print(f"\n  Command: {' '.join(cmd)}")
        print("  (This process takes a significant amount of time)")
        print("  Checkpointing is enabled — resume is possible after interruption.")

        # Optionally run (commented out for safety)
        # result = subprocess.run(cmd, capture_output=True, text=True, cwd=str(PROJECT_ROOT))
        print("\n  ℹ️  Not running automatically for safety.")
        print("     Run the command above directly in the terminal.")

In [ ]:
"""Final quality check."""

print("=" * 70)
print("✅ Final Quality Check")
print("=" * 70)

# Quality gates check
prep_config_loaded = load_yaml_config("configs/data-preparation.yaml")
quality_gates = prep_config_loaded.get("quality_gates", {})

print("Quality gate criteria:")
print(f"  Min acceptance rate: {quality_gates.get('min_acceptance_rate', 0.7):.0%}")
print(f"  Required types: {quality_gates.get('required_types', [])}")
print(f"  Min tool examples: {quality_gates.get('min_tool_examples', 50)}")
print(f"  Min trajectory examples: {quality_gates.get('min_trajectory_examples', 30)}")

if canonical_path.exists():
    print("\nCurrent status:")
    # Re-count
    import json as json_mod
    type_counts = {}
    total = 0
    for cf in canonical_path.glob("*.jsonl"):
        with open(cf) as f:
            for line in f:
                if line.strip():
                    try:
                        rec = json_mod.loads(line)
                        stype = rec.get("sample_type", "unknown")
                        type_counts[stype] = type_counts.get(stype, 0) + 1
                        total += 1
                    except Exception:
                        pass

    print(f"  Total accepted samples: {total}")
    for stype, count in sorted(type_counts.items()):
        print(f"    {stype}: {count}")

    # Check gates
    tool_count = type_counts.get("tool_selection", 0) + type_counts.get("trajectory", 0)
    traj_count = type_counts.get("trajectory", 0)

    gate_results = [
        ("Min tool examples", tool_count >= quality_gates.get("min_tool_examples", 50)),
        ("Min trajectory examples", traj_count >= quality_gates.get("min_trajectory_examples", 30)),
    ]

    for name, passed in gate_results:
        print(f"  {'✅' if passed else '❌'} {name}")
else:
    print("  ⚠️  No generated data found.")

print("\nNext steps:")
print("  📓 03_validate_and_release.ipynb — Data Validation and Bundle Release")